In [9]:
import optuna
import torch
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger
from torchmetrics import F1Score
import warnings
from GradientGang.Pipeline.Architectures.LightningAutoencoder import LightningAutoencoder

import dotenv
import os

warnings.filterwarnings('ignore')

from GradientGang.Pipeline.DataLoader.DataLoader import DataModule
from GradientGang.Pipeline.Architectures.Direct import Direct

# Load database configuration
dotenv.load_dotenv(dotenv_path="./../src/GradientGang/Pipeline/Optimizer/.env")
storage = os.getenv("DATABASE_URL")

print("✓ Database configuration loaded")
print(f"Storage: {storage[:21]}..." if storage else "⚠️ No database URL found")

✓ Database configuration loaded
Storage: postgresql://postgres...


# Mega Optuna Notebook - Multi-Architecture Optimization

This notebook performs comprehensive hyperparameter optimization for the Pirate Pain Classification task.

## Features
- **Database Integration**: Results are stored in a PostgreSQL database for persistence and multi-process optimization
- **Multi-Architecture Search**: Supports Direct and Autoencoder architectures with RNN and Conv1d encoders
- **Smart Pruning**: Uses MedianPruner to stop unpromising trials early
- **Resume Capability**: Can continue optimization from where it left off

## Supported Architectures

### Macro Architectures:
1. **Direct**: End-to-end encoder + classifier
2. **Autoencoder**: Encoder-Decoder with joint reconstruction + classification loss (semi-supervised with test data)

### Encoder Types:
1. **Recurrent (RNN)**: LSTM/GRU with configurable layers, bidirectionality, and dropout
2. **Conv1d**: 1D Convolutional networks with adaptive pooling

### Search Space:
- **Encoder**: Architecture type, hidden dimensions, number of layers, dropout, activation
- **Classifier Head**: Number of layers, hidden dimensions, dropout, activation  
- **Training**: Learning rate, regularization weight, patience, max epochs
- **Autoencoder** (if selected): Reconstruction loss weight

The optimization uses TPE sampler with median pruning for efficient hyperparameter search.

In [10]:
# Fixed data loading parameters
data_params = {
    'data_dir': "../dataset/PirateProcessed/",
    'train_file_name': "pirate_pain_train.csv",
    'train_file_name_labels': "pirate_pain_train_labels.csv",
    'test_file_name': "pirate_pain_test.csv",
    'batch_size': 32,
    'num_workers': 0,
    'val_split': 0.2,
    'shuffle': True,
}

# Initialize data module
dataLoader = DataModule(params=data_params)
dataLoader.setup(stage='fit', includeTestInTrain=True)

trainLoader = dataLoader.train_dataloader()
valLoader = dataLoader.val_dataloader()

print("Data loaders initialized successfully!")
print(f"Training batches: {len(trainLoader)}")
print(f"Validation batches: {len(valLoader)}")

Data loaders initialized successfully!
Training batches: 58
Validation batches: 5


In [11]:
# Display current search space
print("=" * 60)
print("HYPERPARAMETER SEARCH SPACE")
print("=" * 60)
print("\n📊 Macro Architecture:")
print("  • Direct (end-to-end)")
print("  • Autoencoder (semi-supervised with test data)")

print("\n🏗️ Encoder Types:")
print("  • Recurrent: LSTM/GRU")
print("    - Hidden dim: 16-256")
print("    - Layers: 1-3")
print("    - Bidirectional: True/False")
print("    - Dropout: 0.0-0.5")
print("  • Conv1d:")
print("    - Layers: 1-3")
print("    - Kernel size: 3/5/7")
print("    - Channels: 16-128 per layer")
print("    - Stride: 1/2")

print("\n🧠 Classifier Head:")
print("  • Layers: 1-3")
print("  • Hidden dim: 32-256")
print("  • Dropout: 0.0-0.5")
print("  • Activation: ReLU/LeakyReLU/GELU")

print("\n⚙️ Training:")
print("  • Learning rate: 1e-5 to 1e-2 (log scale)")
print("  • Regularization: 1e-6 to 1e-2 (log scale)")
print("  • Patience: 3-15")
print("  • Max epochs: 20-100")

print("\n🔄 Autoencoder (if selected):")
print("  • Reconstruction loss weight: 0.1-0.9")

print("\n" + "=" * 60)

HYPERPARAMETER SEARCH SPACE

📊 Macro Architecture:
  • Direct (end-to-end)
  • Autoencoder (semi-supervised with test data)

🏗️ Encoder Types:
  • Recurrent: LSTM/GRU
    - Hidden dim: 16-256
    - Layers: 1-3
    - Bidirectional: True/False
    - Dropout: 0.0-0.5
  • Conv1d:
    - Layers: 1-3
    - Kernel size: 3/5/7
    - Channels: 16-128 per layer
    - Stride: 1/2

🧠 Classifier Head:
  • Layers: 1-3
  • Hidden dim: 32-256
  • Dropout: 0.0-0.5
  • Activation: ReLU/LeakyReLU/GELU

⚙️ Training:
  • Learning rate: 1e-5 to 1e-2 (log scale)
  • Regularization: 1e-6 to 1e-2 (log scale)
  • Patience: 3-15
  • Max epochs: 20-100

🔄 Autoencoder (if selected):
  • Reconstruction loss weight: 0.1-0.9



In [12]:
def setUpEncoder(trial:optuna.Trial, architectureParameters:dict, datasetInfo:dict):
    # Setup global features encoder
    # Global features is just one binary (isPirate), so we don't need a hidden dimension
    globalEmbeddingDim = 1  # Fixed to 1 since it's just one binary feature
    globalEncoderParams = {
        "activation_function": "LeakyReLU",
        "layer_type": [
            {
                "name": "Linear",
                "params": {
                    "in_features": datasetInfo["globalFeaturesShape"][0],
                    "out_features": globalEmbeddingDim,
                    "bias": True,
                }
            },
        ]
    }
    architectureParameters["GlobalFFEncoderParams"] = globalEncoderParams
    
    # Setup time series encoder - now supports RNN and Conv1d
    architectureType = trial.suggest_categorical("architectureType", ["Recurrent", "Conv1d"])
    
    timeSeriesEncoderParams = {}
    
    if architectureType == "Recurrent":
        rnnType = trial.suggest_categorical("rnnType", ["LSTM", "GRU"])
        hiddenDim = trial.suggest_int("hiddenDim", 16, 256)
        numLayers = trial.suggest_int("numLayers", 1, 3)
        bidirectional = trial.suggest_categorical("bidirectional", [False, True])
        dropout = trial.suggest_float("recurrentDropout", 0.0, 0.5)
        activationFunction = trial.suggest_categorical("encoderActivation", ["ReLU", "LeakyReLU", "GELU"])
        
        # Get input size from dataset info
        inputSize = datasetInfo["timeSeriesShape"][0]
        
        timeSeriesEncoderParams = {
            "activation_function": activationFunction,
            "layer_type": [
                {
                    "name": rnnType,
                    "params": {
                        "input_size": inputSize,
                        "hidden_size": hiddenDim,
                        "num_layers": numLayers,
                        "bias": True,
                        "batch_first": True,
                        "dropout": dropout if numLayers > 1 else 0.0,
                        "bidirectional": bidirectional,
                    }
                },
            ]
        }
        
    elif architectureType == "Conv1d":
        # Conv1d architecture
        numConvLayers = trial.suggest_int("numConvLayers", 1, 3)
        kernelSize = trial.suggest_categorical("kernelSize", [3, 5, 7])
        stride = trial.suggest_categorical("stride", [1, 2])
        activationFunction = trial.suggest_categorical("encoderActivation", ["ReLU", "LeakyReLU", "GELU"])
        
        # Start with input channels
        inputChannels = datasetInfo["timeSeriesShape"][0]
        
        # Build conv layers
        layerList = []
        currentChannels = inputChannels
        
        for i in range(numConvLayers):
            # Increase channels as we go deeper
            outChannels = trial.suggest_int(f"conv{i+1}_channels", 16, 128)
            
            layerList.append({
                "name": "Conv1d",
                "params": {
                    "in_channels": currentChannels,
                    "out_channels": outChannels,
                    "kernel_size": kernelSize,
                    "stride": stride,
                    "padding": kernelSize // 2,  # Same padding
                    "bias": True,
                }
            })
            
            # Add pooling after conv (except last layer)
            if i < numConvLayers - 1:
                poolType = trial.suggest_categorical(f"pool{i+1}_type", ["MaxPool1d", "AvgPool1d"])
                layerList.append({
                    "name": poolType,
                    "params": {
                        "kernel_size": 2,
                        "stride": 2,
                    }
                })
            
            currentChannels = outChannels
        
        # Add global pooling to reduce to fixed size
        layerList.append({
            "name": "AdaptiveAvgPool1d",
            "params": {
                "output_size": 1,
            }
        })
        
        # Flatten
        layerList.append({
            "name": "Flatten",
            "params": {}
        })
        
        timeSeriesEncoderParams = {
            "activation_function": activationFunction,
            "layer_type": layerList
        }
    
    architectureParameters["EncoderParams"] = timeSeriesEncoderParams
    return architectureParameters

In [13]:
def setUpFeedForwardHead(trial:optuna.Trial, architectureParameters:dict, datasetInfo:dict):
    """Setup the feedforward classification head"""
    # Calculate input size based on encoder outputs
    globalEmbeddingDim = architectureParameters["GlobalFFEncoderParams"]["layer_type"][0]["params"]["out_features"]
    
    # Determine encoder output size based on architecture type
    encoderParams = architectureParameters["EncoderParams"]
    
    # Check if this is RNN or Conv1d
    firstLayer = encoderParams["layer_type"][0]
    
    if firstLayer["name"] in ["LSTM", "GRU", "RNN"]:
        # RNN architecture
        hiddenDim = firstLayer["params"]["hidden_size"]
        bidirectional = firstLayer["params"]["bidirectional"]
        rnnOutputSize = hiddenDim * (2 if bidirectional else 1)
        encoderOutputSize = rnnOutputSize
        
    elif firstLayer["name"] == "Conv1d":
        # Conv1d architecture - find the last conv layer before pooling
        lastConvLayer = None
        for layer in encoderParams["layer_type"]:
            if layer["name"] == "Conv1d":
                lastConvLayer = layer
        
        if lastConvLayer:
            # After AdaptiveAvgPool1d(1) and Flatten, output size = out_channels
            encoderOutputSize = lastConvLayer["params"]["out_channels"]
        else:
            encoderOutputSize = 64  # Fallback
    else:
        # Fallback
        encoderOutputSize = 64
    
    combinedInputSize = encoderOutputSize + globalEmbeddingDim
    
    # Suggest feedforward head architecture
    numHiddenLayers = trial.suggest_int("numFFLayers", 1, 3)
    ffHiddenDim = trial.suggest_int("ffHiddenDim", 32, 256)
    ffDropout = trial.suggest_float("ffDropout", 0.0, 0.5)
    ffActivation = trial.suggest_categorical("ffActivation", ["ReLU", "LeakyReLU", "GELU"])
    
    # Build layer list - make sure last element is always Linear
    layerList_clean = []
    currentDim = combinedInputSize
    for i in range(numHiddenLayers):
        layerList_clean.append({
            "name": "Linear",
            "params": {
                "in_features": currentDim,
                "out_features": ffHiddenDim,
                "bias": True,
            }
        })
        # Add dropout BEFORE the next layer (not after the last one)
        if ffDropout > 0 and i < numHiddenLayers - 1:
            layerList_clean.append({
                "name": "Dropout",
                "params": {
                    "p": ffDropout,
                    "inplace": False,
                }
            })
        currentDim = ffHiddenDim
    
    # Note: Final output layer will be added by Direct/Autoencoder class
    # The last layer MUST have "out_features" for Direct to append the output layer
    feedForwardParams = {
        "activation_function": ffActivation,
        "layer_type": layerList_clean
    }
    
    architectureParameters["FeedForwardParams"] = feedForwardParams
    return architectureParameters

In [14]:
def setUpDecoder(trial:optuna.Trial, architectureParameters:dict, datasetInfo:dict):
    """Setup the decoder for autoencoder architecture (mirrors the encoder)"""
    encoderParams = architectureParameters["EncoderParams"]
    firstLayer = encoderParams["layer_type"][0]
    
    # Mirror the encoder architecture
    if firstLayer["name"] in ["LSTM", "GRU", "RNN"]:
        # RNN decoder
        rnnType = firstLayer["name"]
        hiddenDim = firstLayer["params"]["hidden_size"]
        numLayers = firstLayer["params"]["num_layers"]
        bidirectional = firstLayer["params"]["bidirectional"]
        dropout = firstLayer["params"]["dropout"]
        activationFunction = encoderParams["activation_function"]
        
        # Output should reconstruct the input
        outputSize = datasetInfo["timeSeriesShape"][0]
        # Note: seq_len is NOT a parameter for RNN layers - it's handled by the input data shape
        
        decoderParams = {
            "activation_function": activationFunction,
            "layer_type": [
                {
                    "name": rnnType,
                    "params": {
                        "input_size": hiddenDim * (2 if bidirectional else 1),
                        "hidden_size": outputSize,
                        "num_layers": numLayers,
                        "bias": True,
                        "batch_first": True,
                        "dropout": dropout if numLayers > 1 else 0.0,
                        "bidirectional": False,  # Decoder typically not bidirectional
                    }
                },
            ]
        }
        
    elif firstLayer["name"] == "Conv1d":
        # Conv1d decoder (using ConvTranspose1d)
        # Reverse the encoder layers
        layerList = []
        
        # Get conv layers from encoder (in reverse)
        convLayers = [l for l in encoderParams["layer_type"] if l["name"] == "Conv1d"]
        convLayers.reverse()
        
        for i, convLayer in enumerate(convLayers):
            if i == 0:
                inChannels = convLayer["params"]["out_channels"]
            else:
                inChannels = convLayers[i-1]["params"]["in_channels"]
            
            outChannels = convLayer["params"]["in_channels"] if i < len(convLayers) - 1 else datasetInfo["timeSeriesShape"][0]
            
            layerList.append({
                "name": "ConvTranspose1d",
                "params": {
                    "in_channels": inChannels,
                    "out_channels": outChannels,
                    "kernel_size": convLayer["params"]["kernel_size"],
                    "stride": convLayer["params"]["stride"],
                    "padding": convLayer["params"]["padding"],
                    "bias": True,
                }
            })
        
        decoderParams = {
            "activation_function": encoderParams["activation_function"],
            "layer_type": layerList
        }
    else:
        # Fallback
        decoderParams = encoderParams
    
    # Mirror global decoder
    globalEmbeddingDim = architectureParameters["GlobalFFEncoderParams"]["layer_type"][0]["params"]["out_features"]
    globalDecoderParams = {
        "activation_function": "LeakyReLU",
        "layer_type": [
            {
                "name": "Linear",
                "params": {
                    "in_features": globalEmbeddingDim,
                    "out_features": datasetInfo["globalFeaturesShape"][0],
                    "bias": True,
                }
            },
        ]
    }
    
    architectureParameters["DecoderParams"] = decoderParams
    architectureParameters["GlobalFFDecoderParams"] = globalDecoderParams
    return architectureParameters

In [15]:
def objective(trial: optuna.trial.Trial) -> float:
    """
    Objective function for Optuna optimization.
    Returns validation F1 score to maximize.
    """
    
    # Suggest macro architecture
    macroArchitecture = trial.suggest_categorical("MacroArchitecture", ["Direct", "Autoencoder"])
    
    # Setup data (include test for autoencoder, exclude for direct)
    includeTestInTrain = macroArchitecture == "Autoencoder"
    dataLoader.setup(stage='fit', includeTestInTrain=includeTestInTrain)
    trainLoader = dataLoader.train_dataloader()
    valLoader = dataLoader.val_dataloader()
    datasetInfo = dataLoader.getDatasetInfo()
    
    # Build architecture parameters
    archParams = {}
    
    # Setup encoders
    archParams = setUpEncoder(trial, archParams, datasetInfo)
    
    # Setup feedforward head
    archParams = setUpFeedForwardHead(trial, archParams, datasetInfo)
    
    # If autoencoder, setup decoder
    if macroArchitecture == "Autoencoder":
        archParams = setUpDecoder(trial, archParams, datasetInfo)
        # Add reconstruction loss weight
        archParams["ReconstructionLossWeight"] = trial.suggest_float("ReconstructionLossWeight", 0.1, 0.9)
    
    # Add common parameters
    archParams["OutputDim"] = 3  # no_pain, low_pain, high_pain
    archParams["LearningRate"] = trial.suggest_float("LearningRate", 1e-5, 1e-2, log=True)
    archParams["RegularizationWeight"] = trial.suggest_float("RegularizationWeight", 1e-6, 1e-2, log=True)
    archParams["Patience"] = trial.suggest_int("Patience", 3, 15)
    archParams["ClassWeightsPath"] = "../dataset/PirateProcessed/class_weights.yaml"
    
    # Create model based on architecture type
    if macroArchitecture == "Direct":
        model = Direct(archParams)
    else:  # Autoencoder
        model = LightningAutoencoder(archParams)
    
    # Training parameters
    max_epochs = trial.suggest_int("max_epochs", 20, 100)
    
    # Add early stopping callback
    early_stopping_callback = EarlyStopping(
        monitor='val_F1',
        patience=archParams["Patience"],
        mode='max',  # We want to maximize F1 score
        verbose=False
    )
    
    # Save best model checkpoint
    checkpoint_callback = ModelCheckpoint(
        monitor='val_F1',
        mode='max',
        save_top_k=1,
        filename=f'trial-{trial.number}-' + '{epoch:02d}-{val_F1:.3f}',
        verbose=False
    )
    
    # Create trainer
    trainer = Trainer(
        max_epochs=max_epochs,
        enable_progress_bar=False,
        enable_model_summary=False,
        log_every_n_steps=20,
        callbacks=[early_stopping_callback, checkpoint_callback],
        enable_checkpointing=True,
    )
    
    # Train the model
    try:
        trainer.fit(model, trainLoader, valLoader)
        
        # Get best validation F1 from checkpoint callback
        best_f1 = checkpoint_callback.best_model_score.item() if checkpoint_callback.best_model_score is not None else 0.0
        
        # Report for pruning
        trial.report(best_f1, step=trainer.current_epoch)
        
        # Handle pruning
        if trial.should_prune():
            raise optuna.TrialPruned()
        
        return best_f1
        
    except Exception as e:
        print(f"Trial {trial.number} failed with error: {e}")
        return 0.0

## Run Optuna Optimization

Configure and run the hyperparameter search. We'll start with a modest number of trials to validate the setup.

**Benefits of Database Storage:**
- 💾 **Persistence**: Results survive notebook restarts
- 🔄 **Resume**: Continue optimization from where you left off
- 🚀 **Parallel**: Run multiple optimization processes simultaneously
- 📊 **Analysis**: Access results from any notebook or script

In [17]:
# Create Optuna study with database storage
study = optuna.create_study(
    direction='maximize',  # Maximize F1 score
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=5,
        n_warmup_steps=5,
        interval_steps=1
    ),
    study_name='PAOLO_pirate_pain_multi_architecture',
    storage=storage,
    load_if_exists=True  # Resume from existing study if available
)

print("✓ Study created/loaded successfully!")
print(f"Study name: {study.study_name}")
print(f"Sampler: {study.sampler.__class__.__name__}")
print(f"Pruner: {study.pruner.__class__.__name__}")
print(f"Storage: {'Database' if storage else 'In-memory'}")
print(f"Total trials: {len(study.trials)}")
if len(study.trials) > 0:
    try:
        print(f"Best trial so far: {study.best_trial.number}")
        print(f"Best F1 score: {study.best_value:.4f}")
    except Exception:
        print("No best yet")

OperationalError: (psycopg2.OperationalError) connection to server at "aws-1-eu-west-1.pooler.supabase.com" (54.247.26.119), port 5432 failed: FATAL:  MaxClientsInSessionMode: max clients reached - in Session mode max clients are limited to pool_size

(Background on this error at: https://sqlalche.me/e/20/e3q8)

## Optional: Load Existing Study from Database

If you want to analyze results from a previous run without creating a new study, use this cell instead of the one below.

In [17]:
# Load existing study from database (alternative to creating new one)
# Uncomment and run this instead of the cell below if you want to just analyze existing results

# study = optuna.load_study(
#     study_name='pirate_pain_multi_architecture',
#     storage=storage
# )
# 
# print(f"✓ Study loaded from database!")
# print(f"Study name: {study.study_name}")
# print(f"Total trials: {len(study.trials)}")
# if len(study.trials) > 0:
#     print(f"Best F1 score: {study.best_value:.4f}")

In [18]:
# Run optimization
# Start with a small number of trials to validate setup
N_TRIALS = 50  # Increase this for longer runs

print(f"Starting optimization with {N_TRIALS} trials...")
print("This may take a while depending on your hardware.")
print("-" * 60)

study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print("\nOptimization completed!")
print(f"Best trial: {study.best_trial.number}")
print(f"Best F1 score: {study.best_value:.4f}")
print(f"\nBest hyperparameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

Starting optimization with 50 trials...
This may take a while depending on your hardware.
------------------------------------------------------------


  0%|          | 0/50 [00:00<?, ?it/s]

[W 2025-11-12 15:46:00,673] Trial 1 failed with parameters: {'MacroArchitecture': 'Autoencoder', 'architectureType': 'Recurrent', 'rnnType': 'LSTM', 'hiddenDim': 29, 'numLayers': 3, 'bidirectional': True, 'recurrentDropout': 0.010292247147901223, 'encoderActivation': 'ReLU', 'numFFLayers': 1, 'ffHiddenDim': 73, 'ffDropout': 0.15212112147976886, 'ffActivation': 'ReLU', 'ReconstructionLossWeight': 0.5894823157779036, 'LearningRate': 2.621087878265438e-05, 'RegularizationWeight': 1.4742753159914662e-05, 'Patience': 7} because of the following error: TypeError("RNNBase.__init__() got an unexpected keyword argument 'seq_len'").
Traceback (most recent call last):
  File "c:\Polimi\Master\3sem\ANN_challenges\GradientGang\.venv\Lib\site-packages\optuna\study\_optimize.py", line 205, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\Paolo\AppData\Local\Temp\ipykernel_36948\1430260082.py", line 43, in objective
    model = LightningAutoencoder(arc

TypeError: RNNBase.__init__() got an unexpected keyword argument 'seq_len'

## Visualize Results

Now let's analyze the optimization results to understand which hyperparameters had the most impact.

In [ ]:
# Optimization history
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_parallel_coordinate

# Plot optimization history
fig = plot_optimization_history(study)
fig.show()

# Plot parameter importances
fig = plot_param_importances(study)
fig.show()

# Plot parallel coordinate (shows relationship between hyperparameters and objective value)
fig = plot_parallel_coordinate(study)
fig.show()

[W 2025-11-12 15:25:27,880] There are no complete trials.


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

## Study Status and Database Info

Check the current status of trials stored in the database.

In [ ]:
# Check study status from database
print(f"Study name: {study.study_name}")
print(f"Direction: {study.direction}")
print(f"Total trials: {len(study.trials)}")
print(f"Completed trials: {len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])}")
print(f"Failed trials: {len([t for t in study.trials if t.state == optuna.trial.TrialState.FAIL])}")
print(f"Pruned trials: {len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])}")
print(f"Running trials: {len([t for t in study.trials if t.state == optuna.trial.TrialState.RUNNING])}")

if len(study.trials) > 0:
    completed_trials = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    if completed_trials:
        print(f"\n✓ Best trial: {study.best_trial.number}")
        print(f"✓ Best F1 score: {study.best_value:.4f}")
        print(f"\nTop 5 trials:")
        sorted_trials = sorted(completed_trials, key=lambda t: t.value, reverse=True)[:5]
        for i, trial in enumerate(sorted_trials, 1):
            arch = trial.params.get('MacroArchitecture', 'Unknown')
            enc = trial.params.get('architectureType', 'Unknown')
            print(f"  {i}. Trial {trial.number}: F1={trial.value:.4f} | {arch} | {enc}")
    
    print("\n📊 Trial states (last 10):")
    for trial in study.trials[-10:]:
        state_symbol = "✓" if trial.state == optuna.trial.TrialState.COMPLETE else "✗" if trial.state == optuna.trial.TrialState.FAIL else "⊗" if trial.state == optuna.trial.TrialState.PRUNED else "⟳"
        value_str = f"F1={trial.value:.4f}" if trial.value is not None else "N/A"
        print(f"  {state_symbol} Trial {trial.number}: {trial.state.name} | {value_str}")
else:
    print("\n⚠️ No trials found in this study. Run optimization to start!")

## Load and Evaluate Best Model

Load the best checkpoint and evaluate it on the validation set.

In [ ]:
# Reconstruct the best model from best trial parameters
from pytorch_lightning import Trainer
from torchmetrics import ConfusionMatrix
from GradientGang.Pipeline.Architectures.LightningAutoencoder import LightningAutoencoder
import torch

best_params = study.best_params
print("Reconstructing best model with parameters:")
for key, value in best_params.items():
    print(f"  {key}: {value}")

# Create a dummy trial to reuse setup functions
class BestTrial:
    def __init__(self, params):
        self.params = params
    
    def suggest_categorical(self, name, choices):
        return self.params[name]
    
    def suggest_int(self, name, low, high, log=False):
        return self.params[name]
    
    def suggest_float(self, name, low, high, log=False):
        return self.params[name]

best_trial = BestTrial(best_params)

# Reconstruct architecture
archParams = {}
archParams = setUpEncoder(best_trial, archParams, dataLoader.getDatasetInfo())
archParams = setUpFeedForwardHead(best_trial, archParams, dataLoader.getDatasetInfo())

# Check if autoencoder
macroArchitecture = best_params.get('MacroArchitecture', 'Direct')
if macroArchitecture == "Autoencoder":
    archParams = setUpDecoder(best_trial, archParams, dataLoader.getDatasetInfo())
    archParams["ReconstructionLossWeight"] = best_params['ReconstructionLossWeight']

archParams['LearningRate'] = best_params['LearningRate']
archParams['RegularizationWeight'] = best_params['RegularizationWeight']
archParams['Patience'] = best_params['Patience']
archParams['OutputDim'] = dataLoader.getDatasetInfo()['numClasses']
archParams['ClassWeightsPath'] = '../dataset/PirateProcessed/class_weights.yaml'

# Find the best checkpoint path (search in lightning_logs)
import os
import glob

# Get the most recent version directory
log_dirs = sorted(glob.glob('../lightning_logs/version_*'), key=os.path.getmtime, reverse=True)
if log_dirs:
    best_checkpoint_path = None
    for log_dir in log_dirs:
        checkpoints = glob.glob(os.path.join(log_dir, 'checkpoints', '*.ckpt'))
        if checkpoints:
            # Get the checkpoint with best metric in filename
            checkpoints = sorted(checkpoints, key=lambda x: float(x.split('val_F1=')[1].split('.ckpt')[0]) if 'val_F1=' in x else 0.0, reverse=True)
            best_checkpoint_path = checkpoints[0]
            break
    
    if best_checkpoint_path:
        print(f"\nLoading checkpoint: {best_checkpoint_path}")
        
        # Load appropriate model type
        if macroArchitecture == "Direct":
            model = Direct.load_from_checkpoint(best_checkpoint_path, **archParams)
        else:
            model = LightningAutoencoder.load_from_checkpoint(best_checkpoint_path, **archParams)
        
        # Evaluate on validation set
        trainer = Trainer(logger=False, enable_checkpointing=False)
        val_results = trainer.validate(model, datamodule=dataLoader)
        
        print(f"\nValidation Results:")
        print(f"  F1 Score: {val_results[0]['val_F1']:.4f}")
        print(f"  Loss: {val_results[0]['val_loss']:.4f}")
    else:
        print("No checkpoint found in lightning_logs!")
else:
    print("No log directories found!")

## Optional: Generate Submission

If you want to generate predictions for the test set, run this cell.

In [ ]:
# Generate predictions for test set
import pandas as pd

if best_checkpoint_path:
    # Load test data
    test_loader = dataLoader.test_dataloader()
    
    # Set model to eval mode
    model.eval()
    model = model.to('cuda' if torch.cuda.is_available() else 'cpu')
    
    all_predictions = []
    
    with torch.no_grad():
        for batch in test_loader:
            timeSeries, globalFeats, _ = batch
            timeSeries = timeSeries.to(model.device)
            globalFeats = globalFeats.to(model.device)
            
            logits = model(timeSeries, globalFeats)
            predictions = torch.argmax(logits, dim=1)
            all_predictions.extend(predictions.cpu().numpy())
    
    # Create submission dataframe
    submission_df = pd.DataFrame({
        'Id': range(len(all_predictions)),
        'Predicted': all_predictions
    })
    
    # Save submission
    submission_path = f'../Submissions/submission_optuna_general2.csv'
    submission_df.to_csv(submission_path, index=False)
    print(f"Submission saved to: {submission_path}")
    print(f"Total predictions: {len(all_predictions)}")
    print(f"Prediction distribution:")
    print(submission_df['Predicted'].value_counts().sort_index())
else:
    print("No checkpoint loaded, skipping submission generation.")